<a href="https://colab.research.google.com/github/marchedev2002/ia/blob/main/clasificador_milton/Clasificador_Frutas_UTN_Borsato_Milton_modelo_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🍎 Clasificación de Frutas con Redes Neuronales Convolucionales
## Universidad Tecnológica Nacional – Regional Rosario
### Materia: Inteligencia Artificial | PyTorch 2.x

---

## Objetivos
1. Preparar y explorar un dataset de imágenes de frutas
2. Diseñar una CNN desde cero con PyTorch
3. Entrenar, evaluar y visualizar predicciones
4. Comparar resultados entre un dataset controlado y uno realista

## Estrategia: dos datasets, dos mundos

| Etapa | Dataset | Objetivo pedagógico |
|-------|---------|---------------------|
| **Etapa 1** | UTN-IA 2026 · Estudio Visual - Etapa 1 | Introducir CNNs, obtener ~95% accuracy rápido |
| **Etapa 2** | UTN-IA 2026 · Food Classification - Training Set | Generalización, overfitting, data augmentation |

> **Pregunta disparadora:** ¿Por qué la misma arquitectura puede obtener 95% en un dataset y solo 70% en otro?


## Celda 0: Configuración del entorno

In [103]:
import torch

# Verificar disponibilidad de GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memoria: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No se detectó GPU. En Colab: Entorno de ejecución → Cambiar tipo → GPU T4')


Dispositivo: cuda
GPU: Tesla T4
Memoria: 15.6 GB


In [104]:
import os, shutil, random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torchvision.models import resnet18, ResNet18_Weights # Resnet
from sklearn.metrics import classification_report, confusion_matrix

# Semilla para reproducibilidad
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed(SEED)

print(f'PyTorch {torch.__version__} | Torchvision {torchvision.__version__}')
print('✅ Librerías importadas')


PyTorch 2.11.0+cu128 | Torchvision 0.26.0+cu128
✅ Librerías importadas


### FUNCIONES GENERALES

In [105]:
EPOCHS = 20
BATCH_SIZE = 32

def crear_resnet(num_classes):
    weights = ResNet18_Weights.DEFAULT

    model = resnet18(weights=weights)

    # Congelar todas las capas
    for param in model.parameters():
        param.requires_grad = False

    # Reemplazar la capa final
    model.fc = nn.Linear(model.fc.in_features, num_classes)

    return model


def entrenar_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correcto, total = 0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        _, preds = out.max(1)
        total_loss += loss.item() * imgs.size(0)
        correcto   += preds.eq(labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correcto / total


@torch.no_grad()
def evaluar(model, loader, criterion, device):
    model.eval()
    total_loss, correcto, total = 0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out = model(imgs)
        loss = criterion(out, labels)
        _, preds = out.max(1)
        total_loss += loss.item() * imgs.size(0)
        correcto   += preds.eq(labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correcto / total


def descongelar_ultimas_capas(model, num_capas_a_liberar=2):
    # Primero descongelamos todo o seleccionamos específicamente
    for param in model.parameters():
        param.requires_grad = True

    for name, param in model.named_parameters():
        if "layer4" in name or "fc" in name:  # 'layer4' es el último bloque de ResNet
            param.requires_grad = True
        else:
            param.requires_grad = False


@torch.no_grad()
def obtener_predicciones(model, loader, device):
    model.eval()
    preds_all, labels_all = [], []
    for imgs, labels in loader:
        out = model(imgs.to(device))
        _, preds = out.max(1)
        preds_all.extend(preds.cpu().numpy())
        labels_all.extend(labels.numpy())
    return np.array(preds_all), np.array(labels_all)


def graficar_confusion(preds, labels, class_names):
    nombres = [c.split()[0] for c in class_names]
    cm = confusion_matrix(labels, preds)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, None]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    sns.heatmap(cm,      annot=True, fmt='d',    cmap='Blues', xticklabels=nombres, yticklabels=nombres, ax=ax1)
    sns.heatmap(cm_norm, annot=True, fmt='.1%',  cmap='Blues', xticklabels=nombres, yticklabels=nombres, ax=ax2)
    ax1.set_title('Matriz de Confusión (conteos)');   ax1.set_ylabel('Real'); ax1.set_xlabel('Predicho')
    ax2.set_title('Matriz de Confusión (%)');        ax2.set_ylabel('Real'); ax2.set_xlabel('Predicho')
    plt.suptitle('Análisis de Errores', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()



def crear_subset_desde_train(clases, raiz, destino, split_train=0.70, split_val=0.15):
    probe = raiz / clases[0]
    src_base = raiz if probe.exists() else raiz / 'train'

    for clase in clases:
        imgs = sorted([
            p for p in (src_base / clase).glob('**/*')
            if p.suffix.lower() in ('.jpg', '.jpeg', '.png')
        ])
        if not imgs:
            print(f'⚠️  Sin imágenes para clase "{clase}" en {src_base}')
            continue

        random.shuffle(imgs)
        n = len(imgs)
        n_val  = int(n * split_val)
        n_test = int(n * (1 - split_train - split_val))

        splits = {
            'val':   imgs[:n_val],
            'test':  imgs[n_val:n_val + n_test],
            'train': imgs[n_val + n_test:],
        }

        for split_name, subset in splits.items():
            dst = destino / split_name / clase
            dst.mkdir(parents=True, exist_ok=True)
            for img in subset:
                shutil.copy(img, dst / img.name)

        print(f'  {clase}: {len(splits["train"])} train | {len(splits["val"])} val | {len(splits["test"])} test')

    print(f'\n✅ Split creado en {destino}')


def cargar_imagen_limpia(path):
    img = Image.open(path)

    if img.mode in ('P', 'RGBA', 'LA'):
        img = img.convert('RGBA')
        fondo = Image.new('RGB', img.size, (255, 255, 255))
        fondo.paste(img, mask=img.split()[3])  # usar canal alpha como máscara
        return fondo
    return img.convert('RGB')


def predecir_imagen_con_tta(model, img_path, transform, device):
    model.eval()
    img = Image.open(img_path).convert("RGB")
    vistas = [img, img.transpose(Image.FLIP_LEFT_RIGHT),
              img.rotate(10), img.rotate(-10)]
    with torch.no_grad():
        probs = []
        for v in vistas:
            t = transform(v).unsqueeze(0).to(device)
            probs.append(F.softmax(model(t), dim=1))
        prob_promedio = torch.mean(torch.stack(probs), dim=0)
        return prob_promedio.argmax(dim=1).item()


In [106]:
import os

os.environ["KAGGLE_API_TOKEN"] = "KGAT_c8dc4e2dc1eda9c2becc05bd31bb1e8c"

---
# 🍓 ETAPA 2: Dataset UTN-IA 2026
## El desafío de generalización

Usamos el mismo modelo con un dataset con fondos variables e iluminación real.
El modelo obtendrá **peor desempeño**. Esta es la lección más importante:

> **El dataset determina tanto o más que la arquitectura.**

**Dataset:** UTN-IA 2026 · Food Classification - Training Set  
**Link:** https://www.kaggle.com/datasets/geronimoforconi/utn-ia-2026-food-classification-training-set  
**Licencia:** Apache 2.0

> ⚠️ Este es el **dataset oficial de la competencia**. Las imágenes de test tienen fondos variados — el mismo modelo que obtuvo ~95% en Etapa 1 verá aquí cómo la distribución del dataset impacta el rendimiento real.


In [107]:
!kaggle datasets download -d geronimoforconi/utn-ia-2026-food-classification-training-set \
    -p /content/data/fruit_recognition --unzip -q
print("✅ Dataset UTN-IA 2026 descargado")

raiz_r = Path("/content/data/fruit_recognition")


Dataset URL: https://www.kaggle.com/datasets/geronimoforconi/utn-ia-2026-food-classification-training-set
License(s): apache-2.0
✅ Dataset UTN-IA 2026 descargado


## Etapa 2: UTN-IA 2026 · Food Classification - Training Set

In [108]:
import shutil, random
from pathlib import Path

RAIZ_FOOD = Path('/content/data/fruit_recognition')
BASE_DIR_R = Path('/content/data/fruit_recognition_splint/')
CLASES_R   = ['apple', 'banana', 'grapes', 'potato', 'tomato']

# Proporciones del split (deben sumar 1.0)
SPLIT_TRAIN = 0.70
SPLIT_VAL   = 0.15
SPLIT_TEST  = 0.15

crear_subset_desde_train(CLASES_R, RAIZ_FOOD, BASE_DIR_R, SPLIT_TRAIN, SPLIT_VAL)

  apple: 39 train | 7 val | 7 test
  banana: 42 train | 9 val | 9 test
  grapes: 56 train | 11 val | 11 test
  potato: 44 train | 9 val | 9 test
  tomato: 52 train | 10 val | 10 test

✅ Split creado en /content/data/fruit_recognition_splint


## Augmentation

In [109]:
transform_train_resnet_realista = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomApply([transforms.RandomPosterize(bits=3)], p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.15))
])

transform_val_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

### Resnet Augmentation

In [110]:
train_dataset_resnet = ImageFolder(
    str(BASE_DIR_R / 'train'),
    transform=transform_train_resnet_realista,
    loader=cargar_imagen_limpia
)

val_dataset_resnet = ImageFolder(
    str(BASE_DIR_R / 'val'),
    transform=transform_val_test,
    loader=cargar_imagen_limpia
)

test_dataset_resnet = ImageFolder(
    str(BASE_DIR_R / 'test'),
    transform=transform_val_test,
    loader=cargar_imagen_limpia
)



train_loader_resnet = DataLoader(
    train_dataset_resnet,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader_resnet = DataLoader(
    val_dataset_resnet,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader_resnet = DataLoader(
    test_dataset_resnet,
    batch_size=BATCH_SIZE,
    shuffle=False
)


### ResNet

In [111]:
class_names_r = train_dataset_resnet.classes
modelo_resnet_realista = crear_resnet(len(class_names_r))
modelo_resnet_realista = modelo_resnet_realista.to(device)

optimizer_resnet = torch.optim.Adam(
    modelo_resnet_realista.fc.parameters(),
    lr=0.001
)

criterion = nn.CrossEntropyLoss()

mejor_val_loss = float('inf')


for epoch in range(EPOCHS):

    tl, ta = entrenar_epoch(
        modelo_resnet_realista,
        train_loader_resnet,
        criterion,
        optimizer_resnet,
        device
    )

    vl, va = evaluar(
        modelo_resnet_realista,
        val_loader_resnet,
        criterion,
        device
    )

    if vl < mejor_val_loss:
        mejor_val_loss = vl
        torch.save({'epoch':epoch, 'model_state_dict':modelo_resnet_realista.state_dict(), 'val_acc':va}, '/content/resnet18_mejor_realista.pth')

    print(f'{epoch:>6} | {tl:>10.4f} | {ta:>9.1%} | {vl:>10.4f} | {va:>9.1%}')


     0 |     1.5306 |     30.9% |     1.3611 |     52.2%
     1 |     1.1374 |     66.1% |     1.0521 |     63.0%
     2 |     0.8982 |     76.4% |     0.8230 |     78.3%
     3 |     0.7141 |     83.3% |     0.6982 |     80.4%
     4 |     0.6270 |     84.5% |     0.6022 |     84.8%
     5 |     0.5236 |     89.7% |     0.5695 |     82.6%
     6 |     0.4525 |     90.1% |     0.4754 |     89.1%
     7 |     0.4542 |     92.7% |     0.4652 |     89.1%
     8 |     0.4447 |     87.6% |     0.5198 |     82.6%
     9 |     0.3613 |     90.1% |     0.4276 |     87.0%
    10 |     0.3702 |     91.0% |     0.4018 |     91.3%
    11 |     0.3060 |     94.4% |     0.3685 |     91.3%
    12 |     0.2971 |     96.1% |     0.4008 |     87.0%
    13 |     0.3082 |     93.1% |     0.3406 |     89.1%
    14 |     0.2749 |     95.3% |     0.3453 |     91.3%
    15 |     0.3043 |     92.7% |     0.4404 |     89.1%
    16 |     0.2809 |     92.7% |     0.3628 |     87.0%
    17 |     0.2530 |     95.3%

In [112]:
test_loss, test_acc = evaluar(
    modelo_resnet_realista,
    test_loader_resnet,
    criterion,
    device
)

print(test_acc)

0.9347826086956522


### FINE-TUNNING

In [113]:
descongelar_ultimas_capas(modelo_resnet_realista, num_capas_a_liberar=3)  # libera layer4 + fc

optimizer_ft = torch.optim.AdamW([
    {'params': modelo_resnet_realista.layer4.parameters(), 'lr': 1e-5},
    {'params': modelo_resnet_realista.fc.parameters(),     'lr': 1e-4},
], weight_decay=1e-2)

scheduler_ft = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_ft, mode='min', factor=0.5, patience=2
)

EPOCHS_FT = 20
PACIENCIA_ES = 4
mejor_val_loss_ft = float('inf')
epocas_sin_mejora = 0

print(f'{"epoch":>6} | {"train_loss":>10} | {"train_acc":>9} | {"val_loss":>10} | {"val_acc":>9} | lr')
for epoch in range(EPOCHS_FT):
    tl, ta = entrenar_epoch(modelo_resnet_realista, train_loader_resnet, criterion, optimizer_ft, device)
    vl, va = evaluar(modelo_resnet_realista, val_loader_resnet, criterion, device)
    scheduler_ft.step(vl)

    if vl < mejor_val_loss_ft:
        mejor_val_loss_ft = vl
        epocas_sin_mejora = 0
        torch.save({'epoch': epoch, 'model_state_dict': modelo_resnet_realista.state_dict(), 'val_acc': va},
                    '/content/resnet18_finetuned.pth')
    else:
        epocas_sin_mejora += 1

    lr_actual = optimizer_ft.param_groups[0]['lr']
    print(f'{epoch:>6} | {tl:>10.4f} | {ta:>9.1%} | {vl:>10.4f} | {va:>9.1%} | {lr_actual:.1e}')

    if epocas_sin_mejora >= PACIENCIA_ES:
        print(f'Early stopping en epoch {epoch} (sin mejora en {PACIENCIA_ES} epocas)')
        break


 epoch | train_loss | train_acc |   val_loss |   val_acc | lr
     0 |     0.2452 |     91.0% |     0.3220 |     91.3% | 1.0e-05
     1 |     0.1989 |     96.1% |     0.3106 |     91.3% | 1.0e-05
     2 |     0.1559 |     98.3% |     0.2925 |     93.5% | 1.0e-05
     3 |     0.1712 |     97.0% |     0.2793 |     95.7% | 1.0e-05
     4 |     0.1762 |     94.8% |     0.2552 |     95.7% | 1.0e-05
     5 |     0.1580 |     96.6% |     0.2601 |     95.7% | 1.0e-05
     6 |     0.1389 |     97.4% |     0.2320 |     95.7% | 1.0e-05
     7 |     0.1213 |     97.9% |     0.2272 |     95.7% | 1.0e-05
     8 |     0.1593 |     94.8% |     0.2293 |     95.7% | 1.0e-05
     9 |     0.1317 |     97.9% |     0.2270 |     95.7% | 1.0e-05
    10 |     0.0979 |     98.3% |     0.2224 |     95.7% | 1.0e-05
    11 |     0.1230 |     99.1% |     0.2172 |     95.7% | 1.0e-05
    12 |     0.0839 |     99.6% |     0.2118 |     95.7% | 1.0e-05
    13 |     0.1008 |     98.7% |     0.2044 |     95.7% | 1.0e-05


In [114]:
ckpt_ft = torch.load('/content/resnet18_finetuned.pth', map_location=device)
modelo_resnet_realista.load_state_dict(ckpt_ft['model_state_dict'])
print(f'Mejor modelo fine-tuneado cargado (epoch {ckpt_ft["epoch"]}, val_acc={ckpt_ft["val_acc"]:.2%})')

test_loss_ft, test_acc_ft = evaluar(modelo_resnet_realista, test_loader_resnet, criterion, device)
print(f'Test accuracy ANTES del fine-tuning (Fase 1, backbone congelado): {test_acc:.2%}')
print(f'Test accuracy DESPUES del fine-tuning (Fase 2, layer4 descongelada): {test_acc_ft:.2%}')

preds_ft, labels_ft = obtener_predicciones(modelo_resnet_realista, test_loader_resnet, device)
print(classification_report(labels_ft, preds_ft, target_names=[c.split()[0] for c in class_names_r]))
graficar_confusion(preds_ft, labels_ft, class_names_r)

'ckpt_ft = torch.load(\'/content/resnet18_finetuned.pth\', map_location=device)\nmodelo_resnet_realista.load_state_dict(ckpt_ft[\'model_state_dict\'])\nprint(f\'Mejor modelo fine-tuneado cargado (epoch {ckpt_ft["epoch"]}, val_acc={ckpt_ft["val_acc"]:.2%})\')\n\ntest_loss_ft, test_acc_ft = evaluar(modelo_resnet_realista, test_loader_resnet, criterion, device)\nprint(f\'Test accuracy ANTES del fine-tuning (Fase 1, backbone congelado): {test_acc:.2%}\')\nprint(f\'Test accuracy DESPUES del fine-tuning (Fase 2, layer4 descongelada): {test_acc_ft:.2%}\')\n\npreds_ft, labels_ft = obtener_predicciones(modelo_resnet_realista, test_loader_resnet, device)\nprint(classification_report(labels_ft, preds_ft, target_names=[c.split()[0] for c in class_names_r]))\ngraficar_confusion(preds_ft, labels_ft, class_names_r)'


# 🎓 PREGUNTAS PARA LA CLASE:
  1. ¿Por qué el mismo modelo obtiene resultados tan diferentes?
  2. ¿Qué estrategias usarías para mejorar el modelo realista?
  3. ¿Por qué el augmentation más agresivo ayuda más en Etapa 2?
  4. ¿Qué es transfer learning y cómo ayudaría aquí?


---
# 🏆 Submission para la Competencia Kaggle
## UTN-IA 2026: Fruit & Vegetable Classifier

Una vez entrenado el modelo en Etapa 2, generamos el archivo `submission.csv` para subirlo a la competencia.

**Competencia:** https://www.kaggle.com/competitions/utn-ia-2026-fruit-vegetable-classifier

**Formato requerido:**
```
image_id,label
img_0001.jpg,apple
img_0002.jpg,tomato
...
```

**Clases válidas:** `apple`, `banana`, `grapes`, `potato`, `tomato`


In [115]:
# Descargar el zip en /content/data
!mkdir -p /content/data

!kaggle competitions download \
    -c utn-ia-2026-fruit-vegetable-classifier \
    -p /content/data

# Renombrar
!mv /content/data/utn-ia-2026-fruit-vegetable-classifier.zip \
    /content/data/utn_competencia_test.zip

# Crear carpeta destino
!mkdir -p /content/data/test_competencia

# Descomprimir
!unzip -q /content/data/utn_competencia_test.zip \
    -d /content/data/test_competencia

100% 38.9M/38.9M [00:00<00:00, 336MB/s]

replace /content/data/test_competencia/sample_submission.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A


In [116]:
DIR_TEST = Path("/content/data/test_competencia/utn_competencia_test/test_images")
imagenes_test = sorted([p for p in DIR_TEST.rglob("*") if p.suffix.lower() in [".jpg",".jpeg",".png"]])
print(f"✅ {len(imagenes_test)} imágenes de test listas para predecir")


✅ 89 imágenes de test listas para predecir


In [117]:
import pandas as pd
CLASES_COMP = ["apple", "banana", "grapes", "potato", "tomato"]
modelo_submit = modelo_resnet_realista
modelo_submit.eval()

predicciones = []

print(f"🔄 Procesando {len(imagenes_test)} imágenes con TTA...")

for img_path in imagenes_test:
    pred_idx = predecir_imagen_con_tta(modelo_submit, img_path, transform_val_test, device)
    pred_label = class_names_r[pred_idx].lower().strip()

    image_id = os.path.splitext(img_path.name)[0]

    predicciones.append({
        "image_id": image_id,
        "label": pred_label
    })

# 3. Crear DataFrame final
df_sub = pd.DataFrame(predicciones)
print(f"✅ Total de predicciones generadas: {len(df_sub)}")
print(df_sub.head(10))
print("Distribución de predicciones:")
print(df_sub["label"].value_counts())


🔄 Procesando 89 imágenes con TTA...


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


✅ Total de predicciones generadas: 89
    image_id   label
0  img_00000  grapes
1  img_00001  grapes
2  img_00002   apple
3  img_00003  banana
4  img_00004  tomato
5  img_00005  grapes
6  img_00006  potato
7  img_00007  potato
8  img_00008  grapes
9  img_00009  potato
Distribución de predicciones:
label
grapes    21
tomato    20
apple     17
banana    17
potato    14
Name: count, dtype: int64


In [118]:
# Validar que todas las clases predichas son válidas
clases_invalidas = df_sub[~df_sub["label"].isin(CLASES_COMP)]
if len(clases_invalidas) > 0:
    print(f"⚠️ {len(clases_invalidas)} predicciones con clases inválidas:")
    print(clases_invalidas)
else:
    print(f"✅ Todas las {len(df_sub)} predicciones tienen clases válidas")

# Guardar submission.csv
df_sub.to_csv("/content/submission.csv", index=False)
print("📄 submission.csv guardado")
print("  → Subilo en: https://www.kaggle.com/competitions/utn-ia-2026-fruit-vegetable-classifier/submit")


✅ Todas las 89 predicciones tienen clases válidas
📄 submission.csv guardado
  → Subilo en: https://www.kaggle.com/competitions/utn-ia-2026-fruit-vegetable-classifier/submit


---
## Conclusiones

1. **El dataset importa tanto como la arquitectura** — UTN-IA 2026 · Estudio Visual - Etapa 1 (aprox 95%) vs. dataset realista (aprox 75%) con el mismo modelo.
2. **Data Augmentation = regularización** — no solo aumenta datos, enseña robustez al modelo.
3. **Las métricas deben complementarse** — accuracy + F1 + matriz de confusión dan una imagen completa.

## 🎯 Trabajo Práctico: subir el puntaje en la competencia

Con el notebook de clase tenés un baseline. Para el TP, explorá:

- **Transfer Learning** — ResNet18, EfficientNet, ViT preentrenados en ImageNet
- **Data Augmentation agresivo** — MixUp, CutMix, RandAugment
- **Fine-tuning** — descongelar capas del backbone gradualmente
- **Ensemble** — promediar predicciones de varios modelos
- **Test-Time Augmentation (TTA)** — predecir con múltiples crops/flips y promediar

**Métrica:** F1 Score Weighted — penaliza clases minoritarias mal predichas.
**URL competencia:** https://www.kaggle.com/competitions/utn-ia-2026-fruit-vegetable-classifier
